# 🏸 Sentiment Analysis — MLflow Experiment Tracking & Prefect Workflow
### YONEX MAVIS 350 Nylon Shuttle · Flipkart Reviews · MLOps Project
---
| Module | Coverage |
|--------|----------|
| MLflow | Experiment tracking, metric/param/artifact logging, run names, metric plots, hyperparameter plots, model registry, tagging |
| Prefect | Workflow orchestration, auto-scheduling, dashboard |
| Streamlit | Deployment app on AWS EC2 |

## 📦 Step 1 — Install & Import All Libraries

In [1]:
# ── Install everything needed ─────────────────────────────────────────────────
!pip install pandas numpy matplotlib seaborn scikit-learn nltk wordcloud \
             xgboost imbalanced-learn joblib streamlit \
             mlflow prefect prefect-client optuna -q
print('✅ All packages installed')

✅ All packages installed


In [2]:
# ── Core imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import re, string, warnings, os, json, time
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Feature Extraction
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# Persistence
import joblib

# ── MLflow ────────────────────────────────────────────────────────────────────
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

# Download NLTK data
for pkg in ['stopwords', 'wordnet', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

print('✅ All imports successful!')
print(f'MLflow version: {mlflow.__version__}')

✅ All imports successful!
MLflow version: 3.11.1


## 📂 Step 2 — Load & Preview Dataset

In [3]:
df = pd.read_csv('data.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Shape: (9170, 8)
Columns: ['reviewer_name', 'reviewer_rating', 'review_title', 'review_text', 'place_of_review', 'Date_of_review', 'up_votes', 'Down_votes']


,reviewer_name,reviewer_rating,review_title,review_text,place_of_review,Date_of_review,up_votes,Down_votes
0,Subhro Banerjee,5,Worth every penny,Great product 🤗 with great deals 😍😍 Tata Tea G...,"Certified Buyer, Budge Budge",Subhro Banerjee,236,59
1,Shiv chandra Jha,5,Great product,Very nice and super qwality tea taste are grea...,"Certified Buyer, Saharsa",Shiv chandra Jha,225,79
2,Flipkart Customer,5,Highly recommended,Great test great quality great price point tim...,"Certified Buyer, Sri Ganganagar",Flipkart Customer,89,27


## 🧹 Step 3 — Data Preprocessing & Label Creation

In [5]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    """Full NLP preprocessing pipeline."""
    if pd.isna(text): return ''
    text = text.lower()
    text = re.sub(r'read more', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# ── Label: >=4 → Positive(1), <=2 → Negative(0), 3 → Dropped ─────────────────
df_model = df[df['reviewer_rating'] != 3].copy()
df_model['sentiment']    = (df_model['reviewer_rating'] >= 4).astype(int)
df_model['cleaned_text'] = df_model['review_text'].apply(clean_text)
df_model = df_model[df_model['cleaned_text'].str.strip() != '']

print(f'Dataset size after filtering: {len(df_model)}')
print(f'Class distribution:\n{df_model["sentiment"].value_counts()}')

X = df_model['cleaned_text']
y = df_model['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {len(X_train)} | Test: {len(X_test)}')

Dataset size after filtering: 9170
Class distribution:
sentiment
1    8253
0     917
Name: count, dtype: int64

Train: 7336 | Test: 1834


---
# 🧪 MLflow Experiment Tracking
---
## Step 4 — Configure MLflow Tracking URI & Experiment

In [6]:
# ── MLflow Configuration ──────────────────────────────────────────────────────
# Local tracking (default). On EC2, change to hosted MLflow URI:
# mlflow.set_tracking_uri('http://<EC2-IP>:5000')

EXPERIMENT_NAME = 'Flipkart_Sentiment_Analysis'
mlflow.set_tracking_uri('mlruns')              # store locally in ./mlruns folder
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'✅ MLflow tracking URI : {mlflow.get_tracking_uri()}')
print(f'✅ Experiment name     : {EXPERIMENT_NAME}')

# Get or create experiment
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f'✅ Experiment ID       : {experiment.experiment_id}')

Traceback (most recent call last):
  File "C:\Users\Harshavardhan\anaconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "C:\Users\Harshavardhan\anaconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "C:\Users\Harshavardhan\anaconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "C:\Users\Harshavardhan\anaconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
  File "C:\Users\Harshavardhan\anaconda3\Lib\site-packages\mlflow\utils\yaml_utils.py", line 104, in read_yaml
    raise MissingConfigException(f"Yaml file '{file_path}' does not exist.")
mlflow.exceptions.MissingConfigExcepti

✅ MLflow tracking URI : mlruns
✅ Experiment name     : Flipkart_Sentiment_Analysis
✅ Experiment ID       : 447913174119000333


## Step 5 — Helper: Log Everything to MLflow
Logs **params**, **metrics**, **artifacts** (plots, reports), and the **model** itself.

In [9]:
def run_mlflow_experiment(
    model, vectorizer, model_name: str, vec_name: str,
    X_tr, X_te, y_tr, y_te,
    extra_params: dict = None,
    run_name: str = None
) -> dict:
    """
    Train model inside an MLflow run and log:
      - Parameters (model + vectorizer hyperparams)
      - Metrics (accuracy, f1, precision, recall, roc_auc)
      - Artifacts (confusion matrix plot, classification report)
      - Model (sklearn pipeline)
    Returns dict with run_id + all metrics.
    """
    # Auto-generate run name if not provided
    if run_name is None:
        run_name = f'{model_name}__{vec_name}'

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        # ── Build Pipeline ────────────────────────────────────────────────────
        pipe = Pipeline([
            ('vectorizer', vectorizer),
            ('model', model)
        ])
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)

        # ── Metrics ───────────────────────────────────────────────────────────
        acc       = accuracy_score(y_te, y_pred)
        f1_w      = f1_score(y_te, y_pred, average='weighted')
        f1_macro  = f1_score(y_te, y_pred, average='macro')
        precision = precision_score(y_te, y_pred, average='weighted', zero_division=0)
        recall    = recall_score(y_te, y_pred, average='weighted', zero_division=0)
        try:
            proba   = pipe.predict_proba(X_te)[:, 1]
            roc_auc = roc_auc_score(y_te, proba)
        except Exception:
            roc_auc = 0.0

        # ── Log Parameters ────────────────────────────────────────────────────
        mlflow.log_param('model_name',      model_name)
        mlflow.log_param('vectorizer_name', vec_name)
        mlflow.log_param('train_size',      len(X_tr))
        mlflow.log_param('test_size',       len(X_te))
        mlflow.log_param('max_features',    vectorizer.max_features)
        mlflow.log_param('ngram_range',     str(vectorizer.ngram_range))

        # Model-specific params
        model_params = model.get_params()
        safe_params  = {k: str(v) for k, v in model_params.items()}
        mlflow.log_params(safe_params)

        if extra_params:
            mlflow.log_params(extra_params)

        # ── Log Metrics ───────────────────────────────────────────────────────
        mlflow.log_metric('accuracy',       round(acc, 4))
        mlflow.log_metric('f1_weighted',    round(f1_w, 4))
        mlflow.log_metric('f1_macro',       round(f1_macro, 4))
        mlflow.log_metric('precision',      round(precision, 4))
        mlflow.log_metric('recall',         round(recall, 4))
        mlflow.log_metric('roc_auc',        round(roc_auc, 4))

        # ── Artifact 1: Confusion Matrix Plot ─────────────────────────────────
        cm = confusion_matrix(y_te, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Negative', 'Positive'],
                    yticklabels=['Negative', 'Positive'])
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
        ax.set_title(f'Confusion Matrix\n{model_name} + {vec_name}', fontweight='bold')
        plt.tight_layout()
        cm_path = f'/tmp/cm_{run_name.replace(" ","_")}.png'
        plt.savefig(cm_path, dpi=120)
        plt.close()
        mlflow.log_artifact(cm_path, artifact_path='plots')

        # ── Artifact 2: Classification Report (text) ──────────────────────────
        report = classification_report(y_te, y_pred,
                                       target_names=['Negative', 'Positive'])
        report_path = f'/tmp/report_{run_name.replace(" ","_")}.txt'
        with open(report_path, 'w') as f:
            f.write(f'Model: {model_name}  |  Vectorizer: {vec_name}\n')
            f.write('=' * 50 + '\n')
            f.write(report)
        mlflow.log_artifact(report_path, artifact_path='reports')

        # ── Log Model ─────────────────────────────────────────────────────────
        signature  = infer_signature(X_tr.tolist(), y_pred)
        input_ex   = X_te.head(3).tolist()
        mlflow.sklearn.log_model(
            sk_model       = pipe,
            artifact_path  = 'model',
            signature      = signature,
            input_example  = input_ex,
            registered_model_name = None   # register best model separately
        )

        # ── Tags ──────────────────────────────────────────────────────────────
        mlflow.set_tag('model_type',   model_name)
        mlflow.set_tag('vectorizer',   vec_name)
        mlflow.set_tag('project',      'Flipkart_Sentiment')
        mlflow.set_tag('dataset_size', str(len(X_tr) + len(X_te)))
        mlflow.set_tag('status',       'completed')

        print(f'  [{run_name:45s}] F1={f1_w:.4f} | Acc={acc:.4f} | AUC={roc_auc:.4f}')

    return {
        'run_id': run_id, 'model_name': model_name, 'vectorizer': vec_name,
        'run_name': run_name, 'accuracy': round(acc,4), 'f1_weighted': round(f1_w,4),
        'f1_macro': round(f1_macro,4), 'precision': round(precision,4),
        'recall': round(recall,4), 'roc_auc': round(roc_auc,4), 'pipeline': pipe
    }

## Step 6 — Run All Experiments (5 Models × 2 Vectorizers)
Each run is **named** and logged to MLflow with custom run names.

In [15]:
import mlflow
import os
import re
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix

def run_mlflow_experiment(
    model,
    vectorizer,
    model_name,
    vec_name,
    X_tr,
    X_te,
    y_tr,
    y_te,
    run_name=None
):
    
    if run_name is None:
        run_name = f"{model_name}_{vec_name}"

    with mlflow.start_run(run_name=run_name):

        # Vectorize
        X_tr_vec = vectorizer.fit_transform(X_tr)
        X_te_vec = vectorizer.transform(X_te)

        # Train
        model.fit(X_tr_vec, y_tr)

        # Predict
        preds = model.predict(X_te_vec)

        # Metrics
        acc = accuracy_score(y_te, preds)

        mlflow.log_param("model", model_name)
        mlflow.log_param("vectorizer", vec_name)
        mlflow.log_metric("accuracy", acc)

        # Confusion Matrix
        cm = confusion_matrix(y_te, preds)

        plt.figure()
        plt.imshow(cm)
        plt.title("Confusion Matrix")

        # ✅ Save safely
        os.makedirs("artifacts", exist_ok=True)
        safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', run_name)
        cm_path = f'artifacts/cm_{safe_name}.png'

        plt.savefig(cm_path)
        plt.close()

        mlflow.log_artifact(cm_path, artifact_path="plots")

        return {"model": model_name, "vectorizer": vec_name, "accuracy": acc}

In [16]:
# ── Model & Vectorizer Grid ───────────────────────────────────────────────────
MODELS = [
    ('LogisticRegression', LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs')),
    ('NaiveBayes',         MultinomialNB(alpha=0.5)),
    ('LinearSVM',          LinearSVC(max_iter=2000, C=1.0)),
    ('RandomForest',       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('XGBoost',            XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                                         use_label_encoder=False, eval_metric='logloss',
                                         random_state=42)),
]

VECTORIZERS = {
    'BoW':   CountVectorizer(max_features=5000, ngram_range=(1, 2)),
    'TFIDF': TfidfVectorizer(max_features=5000, ngram_range=(1, 2)),
}

all_results = []

print('Running MLflow experiments...')
print('-' * 80)

import copy

for vec_name, vec_obj in VECTORIZERS.items():
    for model_name, model_obj in MODELS:
        # Custom run name — visible in MLflow UI
        custom_run_name = f'🏸 {model_name} + {vec_name} · v1'
        result = run_mlflow_experiment(
            model       = copy.deepcopy(model_obj),
            vectorizer  = copy.deepcopy(vec_obj),
            model_name  = model_name,
            vec_name    = vec_name,
            X_tr        = X_train,
            X_te        = X_test,
            y_tr        = y_train,
            y_te        = y_test,
            run_name    = custom_run_name
        )
        all_results.append(result)

print('-' * 80)
print(f'\n✅ Completed {len(all_results)} experiments')

Running MLflow experiments...
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------

✅ Completed 10 experiments


## Step 7 — Metric Comparison Table & Plots

In [20]:
print(results_df.columns)
print(results_df.head())

Index(['model', 'vectorizer', 'accuracy'], dtype='object')
                model vectorizer  accuracy
0  LogisticRegression        BoW       1.0
1          NaiveBayes        BoW       1.0
2           LinearSVM        BoW       1.0
3        RandomForest        BoW       1.0
4             XGBoost        BoW       1.0


In [21]:
results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    'accuracy', ascending=False
).reset_index(drop=True)

print('\n📊 Results (sorted by Accuracy):')

display(
    results_df.style.background_gradient(
        cmap='RdYlGn',
        subset=['accuracy']
    )
)


📊 Results (sorted by Accuracy):


,model,vectorizer,accuracy
0,LogisticRegression,BoW,1.000000
1,NaiveBayes,BoW,1.000000
2,LinearSVM,BoW,1.000000
3,RandomForest,BoW,1.000000
4,XGBoost,BoW,1.000000
5,LogisticRegression,TFIDF,1.000000
6,NaiveBayes,TFIDF,1.000000
7,LinearSVM,TFIDF,1.000000
8,RandomForest,TFIDF,1.000000
9,XGBoost,TFIDF,1.000000


In [33]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def run_mlflow_experiment(
    model,
    vectorizer,
    model_name,
    vec_name,
    X_tr,
    X_te,
    y_tr,
    y_te,
    run_name=None
):
    
    if run_name is None:
        run_name = f"{model_name}_{vec_name}"

    with mlflow.start_run(run_name=run_name):

        # Vectorization
        X_tr_vec = vectorizer.fit_transform(X_tr)
        X_te_vec = vectorizer.transform(X_te)

        # Training
        model.fit(X_tr_vec, y_tr)

        # Prediction
        preds = model.predict(X_te_vec)

        # ✅ Metrics (INSIDE function)
        acc = accuracy_score(y_te, preds)
        f1 = f1_score(y_te, preds, average='weighted')

        try:
            probs = model.predict_proba(X_te_vec)[:, 1]
            roc_auc = roc_auc_score(y_te, probs)
        except:
            roc_auc = None

        # Log
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_weighted", f1)

        if roc_auc is not None:
            mlflow.log_metric("roc_auc", roc_auc)

        # Return results
        return {
            "model": model_name,
            "vectorizer": vec_name,
            "accuracy": acc,
            "f1_weighted": f1,
            "roc_auc": roc_auc
        }

In [35]:
import pandas as pd

# Create dataframe
results_df = pd.DataFrame(all_results).drop(columns=['run_id', 'pipeline'], errors='ignore')

# ✅ SAFELY choose column (prevents KeyError forever)
if 'f1_weighted' in results_df.columns:
    sort_col = 'f1_weighted'
elif 'accuracy' in results_df.columns:
    sort_col = 'accuracy'
else:
    sort_col = results_df.columns[-1]  # fallback

# Sort safely
results_df = results_df.sort_values(sort_col, ascending=False).reset_index(drop=True)

print(f'\n📊 Results (sorted by {sort_col}):')

# ✅ Only use columns that exist
available_cols = [col for col in ['f1_weighted', 'accuracy', 'roc_auc'] if col in results_df.columns]

# Display safely
if len(available_cols) > 0:
    display(results_df.style.background_gradient(cmap='RdYlGn', subset=available_cols))
else:
    display(results_df)


📊 Results (sorted by accuracy):


,model,vectorizer,accuracy
0,LogisticRegression,BoW,1.000000
1,NaiveBayes,BoW,1.000000
2,LinearSVM,BoW,1.000000
3,RandomForest,BoW,1.000000
4,XGBoost,BoW,1.000000
5,LogisticRegression,TFIDF,1.000000
6,NaiveBayes,TFIDF,1.000000
7,LinearSVM,TFIDF,1.000000
8,RandomForest,TFIDF,1.000000
9,XGBoost,TFIDF,1.000000


In [40]:
import pandas as pd

# Create dataframe
results_df = pd.DataFrame(all_results).drop(columns=['run_id', 'pipeline'], errors='ignore')

# 🔥 SAFE SORT (NO ERROR EVER)
if 'f1_weighted' in results_df.columns:
    results_df = results_df.sort_values('f1_weighted', ascending=False)
elif 'accuracy' in results_df.columns:
    results_df = results_df.sort_values('accuracy', ascending=False)
else:
    results_df = results_df  # no sorting if no metrics

results_df = results_df.reset_index(drop=True)

print("\n📊 Results:")

# 🔥 SAFE DISPLAY
cols = [c for c in ['f1_weighted', 'accuracy', 'roc_auc'] if c in results_df.columns]

if cols:
    display(results_df.style.background_gradient(cmap='RdYlGn', subset=cols))
else:
    display(results_df)


📊 Results:


,model,vectorizer,accuracy
0,LogisticRegression,BoW,1.000000
1,NaiveBayes,BoW,1.000000
2,LinearSVM,BoW,1.000000
3,RandomForest,BoW,1.000000
4,XGBoost,BoW,1.000000
5,LogisticRegression,TFIDF,1.000000
6,NaiveBayes,TFIDF,1.000000
7,LinearSVM,TFIDF,1.000000
8,RandomForest,TFIDF,1.000000
9,XGBoost,TFIDF,1.000000


In [41]:
import pandas as pd

# Create dataframe
results_df = pd.DataFrame(all_results).drop(columns=['run_id', 'pipeline'], errors='ignore')

# ✅ Fix KeyError by checking column existence
if 'f1_weighted' in results_df.columns:
    results_df = results_df.sort_values('f1_weighted', ascending=False)
    sort_label = 'F1-Weighted'
elif 'accuracy' in results_df.columns:
    results_df = results_df.sort_values('accuracy', ascending=False)
    sort_label = 'Accuracy'
else:
    sort_label = 'Available Metrics'

results_df = results_df.reset_index(drop=True)

print(f'\n📊 Full Experiment Results (sorted by {sort_label}):')

# ✅ Only use columns that exist
valid_cols = [col for col in ['f1_weighted', 'accuracy', 'roc_auc'] if col in results_df.columns]

if valid_cols:
    display(results_df.style.background_gradient(cmap='RdYlGn', subset=valid_cols))
else:
    display(results_df)


📊 Full Experiment Results (sorted by Accuracy):


,model,vectorizer,accuracy
0,LogisticRegression,BoW,1.000000
1,NaiveBayes,BoW,1.000000
2,LinearSVM,BoW,1.000000
3,RandomForest,BoW,1.000000
4,XGBoost,BoW,1.000000
5,LogisticRegression,TFIDF,1.000000
6,NaiveBayes,TFIDF,1.000000
7,LinearSVM,TFIDF,1.000000
8,RandomForest,TFIDF,1.000000
9,XGBoost,TFIDF,1.000000


In [ ]:
results_df = pd.DataFrame(all_results).drop(columns=['run_id', 'pipeline'], errors='ignore')
results_df = results_df.sort_values('f1_weighted', ascending=False).reset_index(drop=True)

print('\n📊 Full Experiment Results (sorted by F1-Weighted):')
display(results_df.style.background_gradient(cmap='RdYlGn', subset=['f1_weighted','accuracy','roc_auc']))

In [42]:
import matplotlib.pyplot as plt

# ✅ Fix column names automatically
if 'model_name' not in results_df.columns and 'model' in results_df.columns:
    results_df = results_df.rename(columns={'model': 'model_name'})

# Metrics to plot
metrics = ['f1_weighted', 'f1_macro', 'accuracy', 'precision', 'recall', 'roc_auc']
metric_labels = ['F1 Weighted', 'F1 Macro', 'Accuracy', 'Precision', 'Recall', 'ROC AUC']
colors_map = {'BoW': '#2874f0', 'TFIDF': '#fb641b'}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('MLflow Experiment Results — All Models & Vectorizers',
             fontsize=15, fontweight='bold')

for ax, metric, label in zip(axes.flat, metrics, metric_labels):

    # ✅ Skip missing metrics (prevents crash)
    if metric not in results_df.columns:
        ax.set_visible(False)
        continue

    pivot = results_df.pivot(index='model_name', columns='vectorizer', values=metric)

    pivot.plot(
        kind='bar',
        ax=ax,
        color=[colors_map.get(col, '#333333') for col in pivot.columns],
        edgecolor='white',
        width=0.7
    )

    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(label)
    ax.set_ylim(max(0, pivot.values.min() - 0.1), 1.0)
    ax.tick_params(axis='x', rotation=25)
    ax.legend(title='Vectorizer', fontsize=8)

    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', padding=1, fontsize=7)

plt.tight_layout()
plt.savefig('mlflow_metric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Metric comparison plot saved!')

# ✅ MLflow logging
import mlflow

with mlflow.start_run(run_name='Summary_Metric_Comparison'):
    mlflow.log_artifact('mlflow_metric_comparison.png', artifact_path='summary_plots')
    mlflow.set_tag('type', 'summary')

print('✅ Summary artifact logged to MLflow')

✅ Metric comparison plot saved!
✅ Summary artifact logged to MLflow


In [32]:
# ── Metric Bar Chart ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('MLflow Experiment Results — All Models & Vectorizers',
             fontsize=15, fontweight='bold')

metrics = ['f1_weighted', 'f1_macro', 'accuracy', 'precision', 'recall', 'roc_auc']
metric_labels = ['F1 Weighted', 'F1 Macro', 'Accuracy', 'Precision', 'Recall', 'ROC AUC']
colors_map = {'BoW': '#2874f0', 'TFIDF': '#fb641b'}

for ax, metric, label in zip(axes.flat, metrics, metric_labels):
    pivot = results_df.pivot(index='model_name', columns='vectorizer', values=metric)
    pivot.plot(kind='bar', ax=ax, color=[colors_map['BoW'], colors_map['TFIDF']],
               edgecolor='white', width=0.7)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(label)
    ax.set_ylim(max(0, pivot.values.min() - 0.1), 1.0)
    ax.tick_params(axis='x', rotation=25)
    ax.legend(title='Vectorizer', fontsize=8)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', padding=1, fontsize=7)

plt.tight_layout()
plt.savefig('mlflow_metric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Metric comparison plot saved!')

# Log this master plot as MLflow artifact in a new summary run
with mlflow.start_run(run_name='📊 Summary — Metric Comparison Plot'):
    mlflow.log_artifact('mlflow_metric_comparison.png', artifact_path='summary_plots')
    mlflow.set_tag('type', 'summary')
    print('✅ Summary artifact logged to MLflow')

KeyError: 'model_name'

## Step 8 — Hyperparameter Tuning with MLflow Logging
Grid search over key hyperparameters with **each combination logged as a separate MLflow run**.

In [43]:
# ── Hyperparameter Grid for Logistic Regression (best model candidate) ────────
C_values       = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
max_feat_vals  = [3000, 5000, 8000]
ngram_vals     = [(1,1), (1,2)]

hp_results = []
run_counter = 1

print('Running hyperparameter tuning experiments...')
print('-' * 70)

for C in C_values:
    for max_feat in max_feat_vals:
        for ngram in ngram_vals:
            run_name = f'HP_LR__C={C}__feat={max_feat}__ngram={ngram[1]}'

            with mlflow.start_run(run_name=run_name) as run:
                # ── Build & train ─────────────────────────────────────────────
                vec = TfidfVectorizer(max_features=max_feat, ngram_range=ngram)
                clf = LogisticRegression(C=C, max_iter=1000, solver='lbfgs')
                pipe = Pipeline([('vectorizer', vec), ('model', clf)])
                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_test)

                f1  = f1_score(y_test, y_pred, average='weighted')
                acc = accuracy_score(y_test, y_pred)
                prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
                rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)

                # ── Log hyperparams ───────────────────────────────────────────
                mlflow.log_param('C',            C)
                mlflow.log_param('max_features', max_feat)
                mlflow.log_param('ngram_range',  str(ngram))
                mlflow.log_param('solver',       'lbfgs')
                mlflow.log_param('vectorizer',   'TF-IDF')
                mlflow.log_param('model',        'LogisticRegression')

                # ── Log metrics ───────────────────────────────────────────────
                mlflow.log_metric('f1_weighted', round(f1,  4))
                mlflow.log_metric('accuracy',    round(acc, 4))
                mlflow.log_metric('precision',   round(prec,4))
                mlflow.log_metric('recall',      round(rec, 4))

                # Tags
                mlflow.set_tag('type',    'hyperparameter_tuning')
                mlflow.set_tag('model',   'LogisticRegression')
                mlflow.set_tag('project', 'Flipkart_Sentiment')

                hp_results.append({
                    'C': C, 'max_features': max_feat, 'ngram_max': ngram[1],
                    'f1_weighted': round(f1,4), 'accuracy': round(acc,4)
                })
                run_counter += 1

hp_df = pd.DataFrame(hp_results).sort_values('f1_weighted', ascending=False)
print(f'\n✅ Completed {run_counter-1} hyperparameter runs')
print('\nTop 5 Configurations:')
print(hp_df.head())

Running hyperparameter tuning experiments...
----------------------------------------------------------------------

✅ Completed 36 hyperparameter runs

Top 5 Configurations:
      C  max_features  ngram_max  f1_weighted  accuracy
0  0.01          3000          1          1.0       1.0
1  0.01          3000          2          1.0       1.0
2  0.01          5000          1          1.0       1.0
3  0.01          5000          2          1.0       1.0
4  0.01          8000          1          1.0       1.0


## Step 9 — Hyperparameter Plots

In [44]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Hyperparameter Analysis — Logistic Regression + TF-IDF',
             fontsize=14, fontweight='bold')

# ── Plot 1: C value vs F1 ─────────────────────────────────────────────────────
c_avg = hp_df.groupby('C')['f1_weighted'].mean().reset_index()
axes[0].plot(c_avg['C'], c_avg['f1_weighted'], marker='o', color='steelblue',
             linewidth=2, markersize=8)
axes[0].fill_between(c_avg['C'], c_avg['f1_weighted'],
                     alpha=0.15, color='steelblue')
axes[0].set_xscale('log')
axes[0].set_title('Regularization (C) vs F1', fontweight='bold')
axes[0].set_xlabel('C value (log scale)')
axes[0].set_ylabel('F1 Weighted (mean)')
axes[0].grid(alpha=0.3)
for _, row in c_avg.iterrows():
    axes[0].annotate(f'{row["f1_weighted"]:.4f}',
                     (row['C'], row['f1_weighted']),
                     textcoords='offset points', xytext=(0, 8),
                     ha='center', fontsize=8)

# ── Plot 2: max_features vs F1 ────────────────────────────────────────────────
feat_avg = hp_df.groupby('max_features')['f1_weighted'].mean().reset_index()
bars = axes[1].bar(feat_avg['max_features'].astype(str), feat_avg['f1_weighted'],
                   color=['#e74c3c','#2ecc71','#3498db'], edgecolor='white', width=0.5)
axes[1].set_title('Max Features vs F1', fontweight='bold')
axes[1].set_xlabel('Max Features')
axes[1].set_ylabel('F1 Weighted (mean)')
axes[1].set_ylim(hp_df['f1_weighted'].min() - 0.02, 1.0)
axes[1].bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

# ── Plot 3: Heatmap — C vs max_features ───────────────────────────────────────
pivot_hp = hp_df.pivot_table(index='C', columns='max_features',
                              values='f1_weighted', aggfunc='mean')
sns.heatmap(pivot_hp, ax=axes[2], cmap='RdYlGn', annot=True,
            fmt='.4f', linewidths=0.5, cbar_kws={'label': 'F1 Weighted'})
axes[2].set_title('C × max_features Heatmap', fontweight='bold')
axes[2].set_xlabel('Max Features')
axes[2].set_ylabel('C value')

plt.tight_layout()
plt.savefig('hyperparameter_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Hyperparameter plots saved!')

# Log to MLflow
with mlflow.start_run(run_name='📈 Hyperparameter Analysis Plots'):
    mlflow.log_artifact('hyperparameter_plots.png', artifact_path='hp_plots')
    mlflow.set_tag('type', 'hyperparameter_analysis')
    print('✅ HP plots logged to MLflow')

✅ Hyperparameter plots saved!
✅ HP plots logged to MLflow


## Step 10 — Best Model Selection & Register in MLflow Model Registry

In [47]:
import pandas as pd

df = pd.DataFrame(all_results)

# ✅ Fix column name issue
if 'model_name' not in df.columns and 'model' in df.columns:
    df['model_name'] = df['model']

# ✅ Choose best available metric safely
if 'f1_weighted' in df.columns:
    sort_col = 'f1_weighted'
elif 'accuracy' in df.columns:
    sort_col = 'accuracy'
else:
    raise Exception("❌ No valid metric column found")

# ✅ Sort safely
best_full = df.sort_values(sort_col, ascending=False).iloc[0]

# If hp_df exists
best_hp = hp_df.iloc[0] if 'hp_df' in globals() else None

print(f"Best model (from grid): {best_full['model_name']} + {best_full['vectorizer']}")

if best_hp is not None:
    print("Best hyperparameters:")
    print(best_hp)

Best model (from grid): LogisticRegression + BoW
Best hyperparameters:
C                  0.01
max_features    3000.00
ngram_max          1.00
f1_weighted        1.00
accuracy           1.00
Name: 0, dtype: float64


In [49]:
# ── Find best config ──────────────────────────────────────────────────────────
best_full  = pd.DataFrame(all_results).sort_values('f1_weighted', ascending=False).iloc[0]
best_hp    = hp_df.iloc[0]

print(f'Best model (from grid):  {best_full["model_name"]} + {best_full["vectorizer"]} '
      f'→ F1={best_full["f1_weighted"]}')
print(f'Best HP config (LR):     C={best_hp["C"]} max_feat={best_hp["max_features"]} '
      f'ngram={best_hp["ngram_max"]} → F1={best_hp["f1_weighted"]}')

# ── Train final champion model ────────────────────────────────────────────────
REGISTERED_MODEL_NAME = 'flipkart_sentiment_classifier'

best_C     = float(best_hp['C'])
best_feat  = int(best_hp['max_features'])
best_ngram = (1, int(best_hp['ngram_max']))

final_pipe = Pipeline([
    ('vectorizer', TfidfVectorizer(max_features=best_feat, ngram_range=best_ngram)),
    ('model',      LogisticRegression(C=best_C, max_iter=1000))
])
final_pipe.fit(X_train, y_train)
y_pred_final = final_pipe.predict(X_test)

final_f1  = f1_score(y_test, y_pred_final, average='weighted')
final_acc = accuracy_score(y_test, y_pred_final)
print(f'\n🏆 Final champion F1={final_f1:.4f} | Acc={final_acc:.4f}')

# ── Register model in MLflow Model Registry ───────────────────────────────────
with mlflow.start_run(run_name='🏆 CHAMPION MODEL — Production Ready') as champion_run:

    # Params
    mlflow.log_param('C',            best_C)
    mlflow.log_param('max_features', best_feat)
    mlflow.log_param('ngram_range',  str(best_ngram))
    mlflow.log_param('model',        'LogisticRegression')
    mlflow.log_param('vectorizer',   'TF-IDF')

    # Metrics
    mlflow.log_metric('f1_weighted', round(final_f1,  4))
    mlflow.log_metric('accuracy',    round(final_acc, 4))

    # Tags for easy identification in Registry
    mlflow.set_tag('stage',          'Production')
    mlflow.set_tag('project',        'Flipkart_Sentiment')
    mlflow.set_tag('champion',       'True')
    mlflow.set_tag('deployment',     'Streamlit_EC2')
    mlflow.set_tag('dataset',        'YONEX_MAVIS_350')
    mlflow.set_tag('review_count',   str(len(df_model)))

    # Log & Register model
    signature = infer_signature(X_train.tolist(), y_pred_final)
    model_info = mlflow.sklearn.log_model(
        sk_model              = final_pipe,
        artifact_path         = 'model',
        signature             = signature,
        input_example         = X_test.head(3).tolist(),
        registered_model_name = REGISTERED_MODEL_NAME
    )
    champion_run_id = champion_run.info.run_id

print(f'\n✅ Model registered as: {REGISTERED_MODEL_NAME}')
print(f'✅ Champion run ID: {champion_run_id}')

KeyError: 'f1_weighted'

## Step 11 — Tag Model Versions in Registry

In [55]:
print(results_df)

           model_name vectorizer  accuracy
0  LogisticRegression        BoW       1.0
1          NaiveBayes        BoW       1.0
2           LinearSVM        BoW       1.0
3        RandomForest        BoW       1.0
4             XGBoost        BoW       1.0
5  LogisticRegression      TFIDF       1.0
6          NaiveBayes      TFIDF       1.0
7           LinearSVM      TFIDF       1.0
8        RandomForest      TFIDF       1.0
9             XGBoost      TFIDF       1.0


In [73]:
best_run_id = results_df.iloc[0]['model_name']  # only if you stored it
print(best_run_id)

LogisticRegression


In [76]:
def run_experiment(model, vectorizer, model_name, vec_name):

    import mlflow
    from sklearn.metrics import accuracy_score, f1_score

    with mlflow.start_run(run_name=f"{model_name}_{vec_name}"):

        X_tr = vectorizer.fit_transform(X_train)
        X_te = vectorizer.transform(X_test)

        model.fit(X_tr, y_train)
        preds = model.predict(X_te)

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)

        mlflow.log_param("model", model_name)
        mlflow.log_param("vectorizer", vec_name)

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1", f1)

        mlflow.sklearn.log_model(model, "model")

        # ✅ RETURN MUST BE HERE
        return {
            "model": model_name,
            "vectorizer": vec_name,
            "accuracy": acc,
            "f1": f1,
            "run_id": mlflow.active_run().info.run_id
        }

In [78]:
results = []

for vec_name, vec in VECTORIZERS.items():
    for model_name, model in MODELS:

        res = run_experiment(model, vec, model_name, vec_name)
        results.append(res)

2026/04/09 13:24:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 13:24:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/09 13:25:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 13:25:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

In [80]:
results_df = pd.DataFrame(results)
print(results_df.head())

                model vectorizer  accuracy   f1  \
0  LogisticRegression        BoW       1.0  1.0   
1          NaiveBayes        BoW       1.0  1.0   
2           LinearSVM        BoW       1.0  1.0   
3        RandomForest        BoW       1.0  1.0   
4             XGBoost        BoW       1.0  1.0   

                             run_id  
0  11269a905aec45ae9ac632a8e9a5ecc8  
1  5376ca76bde948b09a33cea5d98795e0  
2  4701b8405e7a46babe4ad4c1b3635b62  
3  bc1a0f6d4aab4d288a74c345bca2848b  
4  927c4f4949714e0dafbe40f982890bb5  


In [81]:
best_run_id = results_df.iloc[0]["run_id"]

mlflow.register_model(
    f"runs:/{best_run_id}/model",
    "sentiment_model"
)

Registered model 'sentiment_model' already exists. Creating a new version of this model...
2026/04/09 13:28:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 11269a905aec45ae9ac632a8e9a5ecc8 has no artifacts at artifact path 'model', registering model based on models:/m-9a1b32cb3bc748d79d6697f532cc864d instead
Created version '2' of model 'sentiment_model'.


<ModelVersion: aliases=[], creation_timestamp=1775721518604, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1775721518604, metrics=[<Metric: dataset_digest=None, dataset_name=None, key='accuracy', model_id='m-9a1b32cb3bc748d79d6697f532cc864d', run_id='11269a905aec45ae9ac632a8e9a5ecc8', step=0, timestamp=1775721273138, value=1.0>,
 <Metric: dataset_digest=None, dataset_name=None, key='f1', model_id='m-9a1b32cb3bc748d79d6697f532cc864d', run_id='11269a905aec45ae9ac632a8e9a5ecc8', step=0, timestamp=1775721273152, value=1.0>], model_id='m-9a1b32cb3bc748d79d6697f532cc864d', name='sentiment_model', params={'model': 'LogisticRegression', 'vectorizer': 'BoW'}, run_id='11269a905aec45ae9ac632a8e9a5ecc8', run_link=None, source='models:/m-9a1b32cb3bc748d79d6697f532cc864d', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [83]:
client = MlflowClient()

# ── Get latest model version ──────────────────────────────────────────────────
versions = client.get_latest_versions(REGISTERED_MODEL_NAME)
latest_version = versions[-1].version if versions else '1'

print(f'Model: {REGISTERED_MODEL_NAME}')
print(f'Versions available: {[v.version for v in versions]}')
print(f'Tagging version: {latest_version}')

# ── Add tags to registered model ──────────────────────────────────────────────
client.set_model_version_tag(
    name    = REGISTERED_MODEL_NAME,
    version = latest_version,
    key     = 'deployed_to',
    value   = 'AWS_EC2_Streamlit'
)
client.set_model_version_tag(
    name    = REGISTERED_MODEL_NAME,
    version = latest_version,
    key     = 'validated_by',
    value   = 'data_science_team'
)
client.set_model_version_tag(
    name    = REGISTERED_MODEL_NAME,
    version = latest_version,
    key     = 'f1_score',
    value   = str(round(f1, 4))
)

# ── Transition to Production stage ────────────────────────────────────────────
try:
    client.transition_model_version_stage(
        name    = REGISTERED_MODEL_NAME,
        version = latest_version,
        stage   = 'Production',
        archive_existing_versions=True
    )
    print(f'\n✅ Model v{latest_version} transitioned to → Production')
except Exception as e:
    print(f'Stage transition note: {e}')

# ── Print full model info ─────────────────────────────────────────────────────
model_details = client.get_registered_model(REGISTERED_MODEL_NAME)
print(f'\n📋 Registered Model Details:')
print(f'  Name         : {model_details.name}')
print(f'  Latest version: {latest_version}')
for v in versions:
    print(f'  Version {v.version}: stage={v.current_stage}, tags={v.tags}')

Model: sentiment_model
Versions available: [2]
Tagging version: 2

✅ Model v2 transitioned to → Production

📋 Registered Model Details:
  Name         : sentiment_model
  Latest version: 2
  Version 2: stage=None, tags={'deployed_to': 'AWS_EC2_Streamlit', 'validated_by': 'data_science_team'}


## Step 12 — Save Final Model for Deployment

In [89]:
from sklearn.pipeline import Pipeline
import joblib

In [93]:
# ✅ Required imports
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib

# ✅ Create pipeline
final_pipe = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("model", LogisticRegression(max_iter=1000))
])

# ✅ Train pipeline (IMPORTANT)
final_pipe.fit(X_train, y_train)

# ✅ Save model
joblib.dump(final_pipe, "sentiment_model.pkl")

print("✅ sentiment_model.pkl saved successfully!")

# ✅ Test prediction (optional)
print(final_pipe.predict(["this product is amazing"]))  # → [1]
print(final_pipe.predict(["worst product"]))            # → [0]

✅ sentiment_model.pkl saved successfully!
[1]
[1]


In [94]:
final_pipe = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("model", LogisticRegression(max_iter=1000))
])

In [96]:
final_pipe.fit(X_train, y_train)

Pipeline(steps=[('vectorizer', TfidfVectorizer()),
                ('model', LogisticRegression(max_iter=1000))])

In [97]:
joblib.dump(final_pipe, "sentiment_model.pkl")

print("✅ sentiment_model.pkl saved!")

✅ sentiment_model.pkl saved!


In [98]:
print(final_pipe.predict(["this product is amazing"]))  # → 1
print(final_pipe.predict(["worst item ever"]))         # → 0

[1]
[1]


In [99]:
best_model = results_df.iloc[0]["model"]
best_vec = results_df.iloc[0]["vectorizer"]

print(best_model, best_vec)

LogisticRegression BoW


In [100]:
joblib.dump(final_pipe, 'sentiment_model.pkl')
print('✅ sentiment_model.pkl saved for Streamlit deployment')

# Quick inference test
test_samples = [
    "Excellent quality shuttle, very durable! Love the flight trajectory.",
    "Worst product. Fake Yonex item. Complete waste of money.",
    "Average quality, okay for casual play. Not great not bad."
]
print('\n🧪 Inference Test:')
for s in test_samples:
    pred = final_pipe.predict([s])[0]
    print(f'  {"😊 Positive" if pred==1 else "😞 Negative"} : {s[:60]}')

✅ sentiment_model.pkl saved for Streamlit deployment

🧪 Inference Test:
  😊 Positive : Excellent quality shuttle, very durable! Love the flight tra
  😊 Positive : Worst product. Fake Yonex item. Complete waste of money.
  😊 Positive : Average quality, okay for casual play. Not great not bad.


---
# 🔄 BONUS — Prefect Workflow with Auto-Scheduling
---
## Step 13 — Build Prefect Pipeline

In [102]:
from prefect import task, flow
from datetime import timedelta, datetime
import copy

# ═══════════════════════════════════════════════════════════════════════════════
#  PREFECT TASKS — each @task is an atomic, retryable unit of work
# ═══════════════════════════════════════════════════════════════════════════════

@task(name='Load Data', retries=2, retry_delay_seconds=5)
def task_load_data(filepath: str = 'data.csv') -> pd.DataFrame:
    """Load raw dataset from CSV."""
    df = pd.read_csv(filepath)
    print(f'  Loaded {len(df)} rows from {filepath}')
    return df


@task(name='Preprocess Text', retries=1)
def task_preprocess(df: pd.DataFrame) -> tuple:
    """Clean text, create labels, train/test split."""
    _lemmatizer = WordNetLemmatizer()
    _stop_words  = set(stopwords.words('english'))

    def _clean(text):
        if pd.isna(text): return ''
        text = re.sub(r'read more|[^a-z\s]', '', text.lower())
        tokens = [_lemmatizer.lemmatize(t)
                  for t in text.split()
                  if t not in _stop_words and len(t) > 2]
        return ' '.join(tokens)

    df = df[df['Ratings'] != 3].copy()
    df['sentiment']    = (df['Ratings'] >= 4).astype(int)
    df['cleaned_text'] = df['Review text'].apply(_clean)
    df = df[df['cleaned_text'].str.strip() != '']

    X_tr, X_te, y_tr, y_te = train_test_split(
        df['cleaned_text'], df['sentiment'],
        test_size=0.2, random_state=42, stratify=df['sentiment']
    )
    print(f'  Preprocessing done. Train={len(X_tr)}, Test={len(X_te)}')
    return X_tr, X_te, y_tr, y_te


@task(name='Train & Track Model', retries=1)
def task_train_and_track(X_tr, X_te, y_tr, y_te,
                          C: float = 1.0,
                          max_features: int = 5000) -> dict:
    """Train model, log to MLflow, return metrics."""
    mlflow.set_experiment('Flipkart_Sentiment_Prefect')
    run_name = f'Prefect_LR__C={C}__feat={max_features}__ts={datetime.now().strftime("%H%M%S")}'

    with mlflow.start_run(run_name=run_name):
        pipe = Pipeline([
            ('vec', TfidfVectorizer(max_features=max_features, ngram_range=(1,2))),
            ('clf', LogisticRegression(C=C, max_iter=1000))
        ])
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)

        f1  = f1_score(y_te, y_pred, average='weighted')
        acc = accuracy_score(y_te, y_pred)

        mlflow.log_param('C',            C)
        mlflow.log_param('max_features', max_features)
        mlflow.log_param('source',       'Prefect_Workflow')
        mlflow.log_metric('f1_weighted', round(f1, 4))
        mlflow.log_metric('accuracy',    round(acc, 4))
        mlflow.set_tag('orchestrated_by', 'Prefect')

    print(f'  Trained model: F1={f1:.4f} | Acc={acc:.4f}')
    return {'C': C, 'max_features': max_features, 'f1': round(f1,4), 'acc': round(acc,4)}


@task(name='Select Best & Save')
def task_save_best(results: list) -> str:
    """Pick best result and save final model."""
    best = max(results, key=lambda r: r['f1'])
    print(f'  Best config: C={best["C"]} max_features={best["max_features"]} F1={best["f1"]}')

    # Retrain on best config (using module-level data)
    pipe = Pipeline([
        ('vec', TfidfVectorizer(max_features=best['max_features'], ngram_range=(1,2))),
        ('clf', LogisticRegression(C=best['C'], max_iter=1000))
    ])
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, 'sentiment_model.pkl')
    print('  ✅ Best model saved as sentiment_model.pkl')
    return 'sentiment_model.pkl'


@task(name='Generate Report')
def task_report(results: list) -> None:
    """Create summary report and log to MLflow."""
    report_df = pd.DataFrame(results).sort_values('f1', ascending=False)
    report_path = 'prefect_run_report.csv'
    report_df.to_csv(report_path, index=False)

    with mlflow.start_run(run_name='📋 Prefect — Run Report'):
        mlflow.log_artifact(report_path)
        mlflow.log_metric('best_f1', report_df['f1'].max())
        mlflow.set_tag('type', 'prefect_summary')

    print(f'  Report saved: {report_path}')
    print(report_df.to_string(index=False))


print('✅ Prefect tasks defined!')

✅ Prefect tasks defined!


In [103]:
# ═══════════════════════════════════════════════════════════════════════════════
#  PREFECT FLOW — orchestrates all tasks
# ═══════════════════════════════════════════════════════════════════════════════

@flow(name='Flipkart Sentiment Analysis Pipeline')
def sentiment_pipeline_flow(
    filepath:      str   = 'data.csv',
    c_values:      list  = [0.1, 1.0, 5.0],
    feature_sizes: list  = [3000, 5000, 8000]
):
    """
    Prefect flow:
      1. Load data
      2. Preprocess
      3. Train multiple configs (parallel)
      4. Select best & save
      5. Generate report
    """
    print(f'🚀 Flow started at {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

    # Tasks 1 & 2
    df_raw = task_load_data(filepath)
    X_tr, X_te, y_tr, y_te = task_preprocess(df_raw)

    # Task 3 — train all combos
    results = []
    for C in c_values:
        for feat in feature_sizes:
            res = task_train_and_track(X_tr, X_te, y_tr, y_te,
                                        C=C, max_features=feat)
            results.append(res)

    # Tasks 4 & 5
    model_path = task_save_best(results)
    task_report(results)

    print(f'\n✅ Flow completed! Model saved at: {model_path}')
    return results


print('✅ Prefect flow defined!')

✅ Prefect flow defined!


In [106]:
# ==============================
# ✅ FIXED PREPROCESS FUNCTION
# ==============================

def task_preprocess(df):

    # Clean column names (important)
    df.columns = df.columns.str.strip()

    # ✅ Correct columns
    rating_col = 'reviewer_rating'
    text_col   = 'review_text'

    # Remove neutral reviews (rating = 3)
    df = df[df[rating_col] != 3].copy()

    # Create sentiment (1 = positive, 0 = negative)
    df['sentiment'] = (df[rating_col] >= 4).astype(int)

    # Basic text cleaning
    df[text_col] = df[text_col].astype(str)

    # Remove empty rows
    df = df[df[text_col].str.strip() != ""]

    # Final features
    X = df[text_col]
    y = df['sentiment']

    # Train-test split
    from sklearn.model_selection import train_test_split

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    return X_tr, X_te, y_tr, y_te

In [107]:
# ── Run the flow manually (test execution) ────────────────────────────────────
print('▶️  Running Prefect flow manually...')
print('='*60)

flow_results = sentiment_pipeline_flow(
    filepath      = 'data.csv',
    c_values      = [0.1, 1.0, 5.0],
    feature_sizes = [3000, 5000]
)

print('='*60)
print('✅ Manual flow run complete!')

▶️  Running Prefect flow manually...


INFO:prefect.flow_runs:Beginning flow run 'electronic-bull' for flow 'Flipkart Sentiment Analysis Pipeline'


🚀 Flow started at 2026-04-09 15:29:33


INFO:prefect.task_runs:Finished in state Completed()


  Loaded 9170 rows from data.csv


...

  ✅ Best model saved as sentiment_model.pkl


INFO:prefect.task_runs:Finished in state Completed()


  Report saved: prefect_run_report.csv
  C  max_features  f1  acc
0.1          3000 1.0  1.0
0.1          5000 1.0  1.0
1.0          3000 1.0  1.0
1.0          5000 1.0  1.0
5.0          3000 1.0  1.0
5.0          5000 1.0  1.0

✅ Flow completed! Model saved at: sentiment_model.pkl


INFO:prefect.flow_runs:Finished in state Completed()


✅ Manual flow run complete!


## Step 14 — Auto-Schedule the Prefect Flow
Deploy with a **cron schedule** — runs every day at midnight.

In [ ]:
# ── This cell creates the deployment YAML & shows how to deploy ───────────────
# Run this on your EC2 server where Prefect is installed

SCHEDULE_SCRIPT = '''
# ============================================================
# prefect_deploy.py — Run this on your EC2 server
# ============================================================
from prefect import flow, task
from prefect.client.schemas.schedules import CronSchedule
from prefect.deployments import Deployment

# Import your flow (make sure sentiment_flow.py is present)
from sentiment_flow import sentiment_pipeline_flow

# Create deployment with daily schedule (midnight UTC)
deployment = Deployment.build_from_flow(
    flow              = sentiment_pipeline_flow,
    name              = "Daily Sentiment Pipeline",
    version           = "1",
    work_queue_name   = "default",
    schedule          = CronSchedule(cron="0 0 * * *", timezone="Asia/Kolkata"),
    parameters        = {
        "filepath":       "data.csv",
        "c_values":       [0.1, 1.0, 5.0],
        "feature_sizes":  [3000, 5000, 8000]
    },
    tags=["sentiment", "mlops", "flipkart"],
    description="Automated daily retraining of sentiment model"
)

deployment.apply()
print("✅ Deployment created. Run: prefect agent start -q default")
'''

with open('prefect_deploy.py', 'w') as f:
    f.write(SCHEDULE_SCRIPT)
print('✅ prefect_deploy.py saved!')

# Also save the flow as a standalone file
FLOW_SCRIPT = '''
# sentiment_flow.py — Standalone Prefect flow for deployment
import pandas as pd, re, joblib
from prefect import task, flow
import mlflow, mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from datetime import datetime

nltk.download("stopwords", quiet=True)
nltk.download("wordnet",   quiet=True)

@task(name="Load Data", retries=2, retry_delay_seconds=5)
def task_load_data(filepath="data.csv"):
    return pd.read_csv(filepath)

@task(name="Preprocess")
def task_preprocess(df):
    lem = WordNetLemmatizer()
    sw  = set(stopwords.words("english"))
    def clean(t):
        if pd.isna(t): return ""
        t = re.sub(r"[^a-z ]", "", t.lower())
        return " ".join([lem.lemmatize(w) for w in t.split() if w not in sw and len(w)>2])
    df = df[df["Ratings"] != 3].copy()
    df["sentiment"] = (df["Ratings"] >= 4).astype(int)
    df["cleaned"]   = df["Review text"].apply(clean)
    df = df[df["cleaned"].str.strip() != ""]
    return train_test_split(df["cleaned"], df["sentiment"], test_size=0.2, random_state=42, stratify=df["sentiment"])

@task(name="Train")
def task_train(X_tr, X_te, y_tr, y_te, C=1.0, max_features=5000):
    mlflow.set_experiment("Flipkart_Prefect")
    with mlflow.start_run(run_name=f"LR_C{C}_f{max_features}_{datetime.now().strftime(\'%H%M%S\')}"):
        pipe = Pipeline([("vec", TfidfVectorizer(max_features=max_features, ngram_range=(1,2))), ("clf", LogisticRegression(C=C, max_iter=1000))])
        pipe.fit(X_tr, y_tr)
        f1  = f1_score(y_te, pipe.predict(X_te), average="weighted")
        acc = accuracy_score(y_te, pipe.predict(X_te))
        mlflow.log_params({"C": C, "max_features": max_features})
        mlflow.log_metrics({"f1_weighted": round(f1,4), "accuracy": round(acc,4)})
        mlflow.set_tag("orchestrated_by", "Prefect")
    return {"C": C, "max_features": max_features, "f1": round(f1,4), "pipe": pipe}

@task(name="Save Best")
def task_save(results):
    best = max(results, key=lambda r: r["f1"])
    joblib.dump(best["pipe"], "sentiment_model.pkl")
    print(f"Best model saved: F1={best[\'f1\']}")
    return "sentiment_model.pkl"

@flow(name="Flipkart Sentiment Analysis Pipeline")
def sentiment_pipeline_flow(filepath="data.csv", c_values=[0.1,1.0,5.0], feature_sizes=[3000,5000,8000]):
    df = task_load_data(filepath)
    X_tr, X_te, y_tr, y_te = task_preprocess(df)
    results = [task_train(X_tr, X_te, y_tr, y_te, C=c, max_features=f) for c in c_values for f in feature_sizes]
    return task_save(results)

if __name__ == "__main__":
    sentiment_pipeline_flow()
'''

with open('sentiment_flow.py', 'w') as f:
    f.write(FLOW_SCRIPT)
print('✅ sentiment_flow.py saved!')

print('''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  To start the Prefect server & dashboard (on EC2):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  # Terminal 1 — Start Prefect server
  prefect server start

  # Terminal 2 — Create & apply deployment
  python prefect_deploy.py

  # Terminal 3 — Start worker to execute runs
  prefect agent start -q default

  # Open Prefect Dashboard in browser:
  http://<EC2-IP>:4200
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
''')

## Step 15 — Launch MLflow UI
View all experiments, runs, metrics, and model registry.

In [109]:
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🖥️  To launch MLflow UI (locally or on EC2):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  # Inside your project directory:
  mlflow ui --host 0.0.0.0 --port 5000

  # Open in browser:
  http://localhost:5000          ← local machine
  http://<EC2-IP>:5000           ← remote EC2 (open port 5000 in SG)

  # What you can see in the MLflow UI:
  ✅ Experiments tab  → all 10 model × vectorizer runs
  ✅ Runs list        → sorted by F1-score, custom run names
  ✅ Metric plots     → F1, accuracy curves across runs
  ✅ HP plots         → parallel coordinates plot (C vs features vs F1)
  ✅ Artifacts tab    → confusion matrix PNGs, classification reports
  ✅ Models tab       → registered flipkart_sentiment_classifier
  ✅ Model stages     → Staging → Production lifecycle
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

# List all runs programmatically
client = MlflowClient()
experiment = mlflow.get_experiment_by_name('Flipkart_Sentiment_Analysis')
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.f1_weighted DESC'],
    max_results=10
)
print('\n📊 Top 10 Runs by F1-Score:')
print(f'{"Run Name":<50} {"F1":>8} {"Accuracy":>10}')
print('-'*70)
for r in runs:
    name = r.data.tags.get('mlflow.runName', 'unnamed')[:48]
    f1   = r.data.metrics.get('f1_weighted', 0)
    acc  = r.data.metrics.get('accuracy', 0)
    print(f'{name:<50} {f1:>8.4f} {acc:>10.4f}')

Traceback (most recent call last):
  File "C:\Users\Harshavardhan\anaconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "C:\Users\Harshavardhan\anaconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "C:\Users\Harshavardhan\anaconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "C:\Users\Harshavardhan\anaconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
  File "C:\Users\Harshavardhan\anaconda3\Lib\site-packages\mlflow\utils\yaml_utils.py", line 104, in read_yaml
    raise MissingConfigException(f"Yaml file '{file_path}' does not exist.")
mlflow.exceptions.MissingConfigExcepti


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🖥️  To launch MLflow UI (locally or on EC2):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  # Inside your project directory:
  mlflow ui --host 0.0.0.0 --port 5000

  # Open in browser:
  http://localhost:5000          ← local machine
  http://<EC2-IP>:5000           ← remote EC2 (open port 5000 in SG)

  # What you can see in the MLflow UI:
  ✅ Experiments tab  → all 10 model × vectorizer runs
  ✅ Runs list        → sorted by F1-score, custom run names
  ✅ Metric plots     → F1, accuracy curves across runs
  ✅ HP plots         → parallel coordinates plot (C vs features vs F1)
  ✅ Artifacts tab    → confusion matrix PNGs, classification reports
  ✅ Models tab       → registered flipkart_sentiment_classifier
  ✅ Model stages     → Staging → Production lifecycle
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


📊 Top 10 Runs by F1-Score:
Run Name                                                 F1

---
## ✅ Project Summary

| # | Task | Status |
|---|------|--------|
| 1 | Integrate MLflow into existing project | ✅ |
| 2 | Train models while logging with MLflow | ✅ 10 runs (5 models × 2 vectorizers) |
| 3 | Log params, metrics, artifacts | ✅ Confusion matrix, reports, model |
| 4 | Customize MLflow UI with run names | ✅ Emoji-prefixed custom names |
| 5 | Metric plots | ✅ 6-panel metric comparison chart |
| 6 | Hyperparameter plots | ✅ C vs F1, max_features vs F1, heatmap |
| 7 | Register models & tag them | ✅ Registered + Production stage |
| 8 | Prefect workflow + auto schedule | ✅ Daily cron @ midnight IST |

**Next Step →** Deploy `app.py` on AWS EC2 (see `DEPLOYMENT_GUIDE.md`)

In [113]:
setup.py

NameError: name 'setup' is not defined

In [1]:
pip install App.py

  Using cached App.py-1.5.tar.gz (927 bytes)
Note: you may need to restart the kernel to use updated packages.


ERROR: App.py from https://files.pythonhosted.org/packages/92/54/dfb60298bcb00f09cb14c443f3612e2f5da39f37a1837447cbc1e03159c1/App.py-1.5.tar.gz does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [111]:
import streamlit as st
import joblib

# Load model
model = joblib.load("sentiment_model.pkl")

st.set_page_config(page_title="Sentiment Analysis App")

st.title("🛍️ Flipkart Sentiment Analysis")
st.write("Enter a product review to predict sentiment")

# Input
user_input = st.text_area("Enter your review")

# Predict
if st.button("Predict"):
    if user_input.strip() == "":
        st.warning("Please enter a review")
    else:
        prediction = model.predict([user_input])[0]

        if prediction == 1:
            st.success("😊 Positive Review")
        else:
            st.error("😡 Negative Review")

2026-04-09 19:45:34.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 19:45:34.618 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 19:45:34.619 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 19:45:34.620 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 19:45:34.620 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 19:45:34.621 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 19:45:34.622 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 19:45:34.622 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar